Probabilistic Roadmap Planners (PRM)
====================================

**Topic**: Introduction to PRM planners

Version | Author
------------ | -------------
0.2 | Björn Hein


This worksheet **explains the basic concepts of path planning using a probabilistic roadmap approach (PRM)**

License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (please check: http://creativecommons.org/licenses/by-nc/4.0/)

This worksheet should include all necessary ingredients to implement a basic PRM algorithm.

**Prerequisite**:

* Please read the notebook IP-3-0-Introduction-to-NetworkX for understanding the networkx graph library and its use
* and the notebook IP-4-0-Test-Environments for the collision checker and the benchmarks

**Dependencies**:

* Be aware, you will need the files: IPEnvironment.py (the CollisionChecker is defined there), IPBenchmark.py and IPTestSuite.py (benchmarks), IPPerfMonitor.py (will be discussed later) and IPVISPRMBasics.py (figures of this notebook)

Remember that:

* you have to press ctrl-return or shift-return, to execute the code in the code sections, only then the variables are "generated" and can be used
* you can execute the whole notebook by "Run All"

# Why probabilistic

* Existing **complete algorithms** become practically useless unless DOF is low (between 3 and 6).
  * Computation complexity for high-dimensional configuration spaces grows exponentially
  * Complete, general purpose algorithms are at best exponential.
* Complete solutions can hardly be used for rigid objects translating and rotating in 3D.
* Complete solutions can only used for 6-DOF manipulator arms when using “complicated-hard-to-implement” extensions.


**General Idea**: Trade a **limited amount of completeness** against a **major gain in computing efficiency**.

With PRMs, we only get one glimpse of C-Space.


# PRM - Principle

PRMs search takes place in implicit C-Space (C). The planner consists of two phases:

  * **Learning Phase**
    * Create randomly chosen [or in other sophisticated ways] configurations in C-space
    * Connect configurations by a local planner
    * Create Graph
      * each configuration is a node
      * a path between two configurations is an edge between corresponding nodes
  * A **Query Phase** 
    * Find a path from the start and goal configurations to two nodes of the roadmap
    * Search the graph to find a sequence of edges connecting those nodes in the roadmap --> collision free path
    

## Learning Phase

1. **Start** with an **empty Graph**: $G = (V, E)$
2. **Repeatedly** generate a random **free configuration** and **add** it to V
3. **For every new node** (vertex), $c$, select a number of nodes $n_i$ from $V$ and try to connect $c$ to each of them using a line test.
4. **If a path is found** between $c$ and the selected node $n$, **the edge** $(c,n)$ is added to $E$. 

**Remark**: There are now plenty of ways to change/adapt this algorithm. Therefore there are a huge number of publications about this algorithm.


## Basic - PRM - Algorithm

Graph: $G(V,E)$

1. $V \leftarrow \emptyset$ and $E \leftarrow \emptyset$
2. Loop
   
   1. $c \leftarrow$ a **randomly** chosen **free configuration** 
   2. $V_c \leftarrow$ a set of candidate neighbors of $c$ chosen from $V$
   3. $V \leftarrow V \cup \{c\}$   
   4. **ForAll** $n \in V_c$ in order of increasing distance from $c$ and $n$ do
      
      * **if** ( not in same_connected_component(c,n) **AND** line between $c$ and $n$ is collision free) **then**
      
        * $E \leftarrow E \cup \{(c,n)\}$
        * update G's connected components

### Example

The following figures show a Basic PRM on a scene similar to the one of the lecture slides: radius 4, at most 3 neighbours per new node, 40 nodes. Every step of the learning phase is recorded (see `IPVISPRMBasics.py`); the algorithm is the same as the one implemented later in this notebook.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

from IPEnvironment import CollisionChecker
import IPVISPRMBasics as vis

scene, limits = vis.lectureScene()
exampleChecker = CollisionChecker(scene, limits=limits)
exampleRadius = 4.0

# learning phase with recording of every step: the roadmap and the list of events
exampleRoadmap, exampleEvents = vis.traceBasicPRM(exampleChecker, numNodes=40, radius=exampleRadius,
                                                  maxNeighbours=3, seed=8)

#### Set-up

The roadmap is empty: $G = (V, E)$ with $V = \emptyset$ and $E = \emptyset$.

In [ ]:
ax = vis.plotEnvironment(exampleChecker, r"empty roadmap $G = (\emptyset, \emptyset)$")

#### Generate randomly free configuration

1. Generate a vertex $c$ depending on a chosen sampling strategy (Important parameter regarding planning time).

   * "Basic-PRM" uses uniform sampling over the given limits of every dimension of $C$ using the uniform probability distribution over this interval.

* Check whether configuration $c$ is "free"
* If $c$ is collision free, add $c$ to $V$, otherwise discard.

In [ ]:
vis.plotSituation(exampleChecker, exampleEvents, "first", exampleRadius, legend=True)

### Find neighbours $V_c$

* $V_c = \{ c' \in V | D(c,c') \leq maxdist\}$, $V_c$: all nodes, that are in a given distance defined by a distance-function $D$ (dashed circle).
* $D$ is typically the Euclidean distance of node $c$ and node $c'$
* In this example at most the 3 nearest nodes are used.

As there are no nodes nearby (the circle is empty), $V_c$ is empty and the next node is generated:

In [ ]:
vis.plotSituation(exampleChecker, exampleEvents, "noNeighbours", exampleRadius)

This repeats until there are nodes nearby the new generated node:

In [ ]:
vis.plotSituation(exampleChecker, exampleEvents, "neighbours", exampleRadius)

In increasing distance it is tested, whether the newly generated node can be connected collision-free to the nodes stored in $V_c$. If yes, the edge $(c,c')$ is added to $E$: $E \leftarrow E \cup \{(c,c')\}$ and the connected components are updated (light areas):

In [ ]:
vis.plotSituation(exampleChecker, exampleEvents, "added", exampleRadius)

If the line between $c$ and a neighbour is in collision, no edge is added:

In [ ]:
vis.plotSituation(exampleChecker, exampleEvents, "collision", exampleRadius)

Continue generating nodes and connecting them. A new edge can also connect two components; they are merged into one:

In [ ]:
vis.plotSituation(exampleChecker, exampleEvents, "merge", exampleRadius)

But besides checking for a collision-free connection (this is only one part of the if-clause), **take care of the second condition (s. if-clause in D.)**: if the neighbour is already in the same connected component as $c$, the connection is not even tested. It would not improve the connectivity of the roadmap.

In [ ]:
vis.plotSituation(exampleChecker, exampleEvents, "sameComponent", exampleRadius)

Continue this algorithm (learning-phase) until a given criterion is reached (e.g. number of nodes to be generated randomly). At the end - hopefully - a good roadmap is generated.

In [ ]:
vis.plotSituation(exampleChecker, exampleEvents, "last", exampleRadius)

**Try it** (optional, needs `ipywidgets`): step through the whole learning phase.

In [ ]:
from ipywidgets import interact, IntSlider

interact(lambda step: vis.plotStep(exampleChecker, exampleEvents, step, exampleRadius),
         step=IntSlider(min=0, max=len(exampleEvents) - 1, value=0));

#### Query phase

Now that the roadmap is generated, we are able to search a collision free path on that roadmap.

Therefore the start and the goal positions are added to the graph and it is tried to connect them to nodes on the graph (typically the nearest one; if not possible looking in increasing distance): start $s$ and goal $g$ are connected to the nearest nodes $\hat{s}$ and $\hat{g}$ on $G$.

Then a path is searched in the roadmap using a graph search algorithm, e.g. Dijkstra or $A^*$ (A-star), and the path is collected.

In [ ]:
exampleStart, exampleGoal = [1.5, 3.0], [21.0, 9.5]
queryGraph, queryPath = vis.showQueryPhase(exampleChecker, exampleRoadmap, exampleStart, exampleGoal, exampleRadius)

**Remark**: The algorithm will not find the shortest path! As the Basic PRM never connects two nodes of the same component, the roadmap contains no cycles (it is a forest): there is exactly one path between two nodes, often a detour. On the right the same nodes, but connected with every collision-free edge within the radius: the path is much shorter, but many more collision checks are needed to build this roadmap (see IP-5-2 for a variant).

In [ ]:
vis.showNotShortest(exampleChecker, exampleRoadmap, exampleStart, exampleGoal, exampleRadius)

Setting up a basic PRM-Algorithm
===============================

Import modules
--------------

following modules will be needed to run this notebook, especially two new modules are used **random** and **scipy.spatial.distance**:

In [ ]:
import networkx as nx  # library for handling graphs (please check IP-3-0-Introduction-to-NetworkX)
import random  # possibility to create random numbers (the "P" in PRM)
import numpy as np
import math

# sort nearest neighbour brute force
import heapq

# reduce coding effort by using function provided by scipy
from scipy.spatial.distance import euclidean

Needed functionality
--------------------

* Random position generator (Sampling strategy)
* Computation of **nearest neighbours** for a given node
* Test of connected components
* Collision test (point and **line**): Please take care, we also need a **line test**.


Random position generator
-------------------------

Generate random position in configuration space.

In [ ]:
dim = 2
# random.uniform(a, b) returns a random number in [a, b] (uniform distribution); here [-90, 90],
# normally the limits of the planning space are used, e.g. the limits of the joints of the robot
print("A random number", random.uniform(-90, 90))

pos = [random.uniform(-90, 90) for i in range(dim)]
print("A random position", pos)

Nearest Neighbours
------------------

Find the nearest neighbours in a given distance (= radius) in increasing order of the distance, but at most `maxNeighbours` of them (in the example above 3).

### Brute Force

The examples are using the graph library NetworkX. For deeper information, please consult the notebook IP-3-0-Introduction-to-NetworkX or check https://networkx.org.

In [ ]:
def nearestNeighboursX(pos, graph, radius, maxNeighbours=3):
    """ Brute Force method to find the nodes of a graph near the given position **pos**
    within the distance **radius** in **increasing order** of the distance.
    At most **maxNeighbours** nodes are returned.
    Result: list of (distance, (node, attributes))"""

    heap = list()
    for node in graph.nodes(data=True):  # using (data=True) will generate a list of nodes with all attributes
        distance = euclidean(node[1]['pos'], pos)
        if distance < radius:
            # use a heap-queue to sort the nodes in increasing order
            heapq.heappush(heap, (distance, node))

    result = list()
    while len(heap) > 0 and len(result) < maxNeighbours:
        result.append(heapq.heappop(heap))
    return result

Test with the roadmap of the example above: the found neighbours are numbered by increasing distance.

In [ ]:
newPos = [9.0, 5.0]
result = nearestNeighboursX(newPos, exampleRoadmap, radius=4.0)
for distance, (node, attributes) in result:
    print(f"node {node}: distance {distance:.2f}")

vis.showNeighbours(exampleChecker, exampleRoadmap, newPos, 4.0, result)

Check connectivity
------------------

As previously described, we need the functionality to test whether two nodes are in the same connected component (see the figure "neighbour already in the same component" above). Such a connection should not be tested.

In [ ]:
def inSameConnectedComponent(node1, node2, graph):
    """ Check whether two nodes are part of the same connected component using
        functionality from NetworkX
    """
    for connectedComponent in nx.connected_components(graph):
        if (node1 in connectedComponent) and (node2 in connectedComponent):
            return True

    return False

In [ ]:
G_graph = nx.Graph()
G_graph.add_node("A", pos=[0, 1])
G_graph.add_node("B", pos=[10, 1])
G_graph.add_node("C", pos=[1, 10])
G_graph.add_node("D", pos=[1, 12])
G_graph.add_node("E", pos=[12, 12])

G_graph.add_edge("A", "B")
G_graph.add_edge("B", "C")
G_graph.add_edge("D", "E")

pos = nx.get_node_attributes(G_graph, 'pos')
fig, ax = plt.subplots(figsize=(4, 4))
nx.draw(G_graph, pos=pos, ax=ax, with_labels=True, node_color=vis.NODE_COLOR, edgecolors=vis.EDGE_COLOR)

In [ ]:
path = nx.shortest_path(G_graph, "A", "C")
print(path)

for comp in nx.connected_components(G_graph):
    print(comp, "- is A in this component?", "A" in comp)

print("A and C in same component:", inSameConnectedComponent("A", "C", G_graph))
print("A and E in same component:", inSameConnectedComponent("A", "E", G_graph))

Basic Approach
==============

As discussed in the lecture, the basic PRM consists of two phases: **Learning Phase** and the **Query Phase**. In the Learning Phase a roadmap is generated. In the Query Phase the start position and the goal position are connected to that roadmap and a path on this roadmap is searched.

Learning Phase
--------------
In the Learning Phase the roadmap is generated.

### Simple approach
In a straight forward approach the roadmap is constructed by

1. randomly generating free nodes
2. connect these free nodes to nodes of the roadmap in a certain distance or/and restrict this to certain amount of nodes from the roadmap

Generate **random free** nodes: uniformly distributed samples are generated, the ones in collision are discarded. Following function can be defined

In [ ]:
def getRandomFreePosition(collChecker):
    limits = collChecker.getEnvironmentLimits()
    pos = [random.uniform(limit[0], limit[1]) for limit in limits]
    while collChecker.pointInCollision(pos):
        pos = [random.uniform(limit[0], limit[1]) for limit in limits]
    return pos

In [ ]:
# uniform samples in the example scene: free ones are kept, the others are discarded
ax = vis.plotEnvironment(exampleChecker, "uniform samples: free (kept) and in collision (discarded)")
for i in range(80):
    sample = [random.uniform(limit[0], limit[1]) for limit in exampleChecker.getEnvironmentLimits()]
    if exampleChecker.pointInCollision(sample):
        ax.scatter(*sample, marker="x", s=30, color=vis.COLLISION_COLOR)
    else:
        ax.scatter(*sample, s=20, color=vis.SAMPLE_COLOR)
plt.show()

In [ ]:
def learnRoadmapNearestNeighbour(graph, radius, numNodes, collisionChecker, maxNeighbours=3, collisionCheckingSteps=40):
    """Learning phase: add numNodes random free nodes to the graph. Every new node is connected to its nearest
    neighbours (at most maxNeighbours within radius), if they are not yet in the same connected component and
    the line between them is collision-free."""
    for nodeID in range(1, numNodes + 1):

        # Generate a 'randomly chosen, free configuration'
        pos = getRandomFreePosition(collisionChecker)

        # Find set of candidates to connect to sorted by distance
        result = nearestNeighboursX(pos, graph, radius, maxNeighbours)

        # check connection
        graph.add_node(nodeID, pos=pos)
        for distance, (neighbourID, attributes) in result:
            neighbourPos = attributes['pos']
            if not inSameConnectedComponent(nodeID, neighbourID, graph):
                if not collisionChecker.lineInCollision(pos, neighbourPos, steps=collisionCheckingSteps):
                    graph.add_edge(nodeID, neighbourID, weight=distance)

Query Phase
------------

In the query phase start and goal are connected to the roadmap - if possible ...

In [ ]:
def findPath(graph, start, goal, collisionChecker, radius, maxNeighbours=3, collisionCheckingSteps=40):
    """Query phase: connect start and goal to the nearest roadmap node, which can be reached collision-free,
    and search the shortest path in the graph. Raises an exception of NetworkX if no path is found."""
    for name, position in (("start", start), ("goal", goal)):
        for distance, (neighbourID, attributes) in nearestNeighboursX(position, graph, radius, maxNeighbours):
            if not collisionChecker.lineInCollision(position, attributes['pos'], steps=collisionCheckingSteps):
                graph.add_node(name, pos=position)
                graph.add_edge(name, neighbourID, weight=distance)
                break
    # find shortest weighted path on graph
    path = nx.shortest_path(graph, "start", "goal", weight="weight")
    # return nodelist
    return path

Tests
-----

Following a function for:

* Visualizing the graph
* the obstacles
* the solution

In [ ]:
def basicPRMVisualize(graph, collChecker, solution, nodeSize=40, ax=None):
    # get a list of positions of all nodes by returning the content of the attribute 'pos'
    pos = nx.get_node_attributes(graph, 'pos')

    collChecker.drawObstacles(ax)

    # draw the roadmap
    nx.draw_networkx_edges(graph, pos, ax=ax, edge_color=vis.EDGE_COLOR, width=1.0)
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_size=nodeSize, node_color=vis.NODE_COLOR,
                           edgecolors=vis.EDGE_COLOR, linewidths=0.5)

    # draw the solution path
    pathEdges = list(zip(solution[:-1], solution[1:]))
    nx.draw_networkx_edges(graph, pos, ax=ax, edgelist=pathEdges, edge_color=vis.PATH_COLOR, width=4.0)

    # draw start and goal
    for name, color in (("start", vis.START_COLOR), ("goal", vis.GOAL_COLOR)):
        if name in graph.nodes():
            nx.draw_networkx_nodes(graph, pos, nodelist=[name], ax=ax, node_size=max(3 * nodeSize, 80),
                                   node_color=color, edgecolors="white")

First a test with the example scene from above. With the same seed for the random generator, the implementation generates exactly the roadmap of the example.

In [ ]:
random.seed(8)
G = nx.Graph()
learnRoadmapNearestNeighbour(G, radius=4.0, numNodes=40, collisionChecker=exampleChecker)
print("Same roadmap as in the example:", set(map(frozenset, G.edges())) == set(map(frozenset, exampleRoadmap.edges())))

solution = findPath(G, exampleStart, exampleGoal, exampleChecker, radius=4.0)
print(solution)

fig, ax = plt.subplots(figsize=(8, 4.5))
vis.setupAxes(ax, exampleChecker, "Result of the query phase")
basicPRMVisualize(G, exampleChecker, solution, ax=ax)

# Benchmarking

The benchmarks of `IPTestSuite.py` (see IP-4-0-Test-Environments).

In [ ]:
import IPTestSuite as ts

radius = 20.0
numNodes = 900
collisionCheckingSteps = 40

fig, axes = plt.subplots(2, 2, figsize=(11, 11))
for ax, benchmark in zip(axes.flat, ts.benchList):
    collChecker = benchmark.collisionChecker
    G = nx.Graph()
    learnRoadmapNearestNeighbour(G, radius, numNodes, collChecker, collisionCheckingSteps=collisionCheckingSteps)
    try:
        solution = findPath(G, benchmark.startList[0], benchmark.goalList[0], collChecker, radius,
                            collisionCheckingSteps=collisionCheckingSteps)
        result = f"path with {len(solution)} nodes"
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        solution = []
        result = "no path found"
    print(f"{benchmark.name}: {result} ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")

    vis.setupAxes(ax, collChecker, f"{benchmark.name} (level {benchmark.level}): {result}")
    basicPRMVisualize(G, collChecker, solution, nodeSize=10, ax=ax)

Experiments
-------------

* Try different radius
* Try different size of roadmap (`numNodes`)
* Try different `maxNeighbours` and look at the structure of the roadmap
* Use `random.seed(...)` to get reproducible roadmaps and compare the parameters on the same samples
* How many nodes are needed for the benchmark "Bottleneck" to find a path?